# CHB–MIT linear forecasting versus R4

Compare the direct linear baseline with the fixed R4 reference using existing outputs only. No models are fitted and no original result files are changed.

The notebook checks available negative-window and positive-event IDs, training/validation counts, and recorded model settings before comparing cases. It cannot establish historical training-window identity when the reservoir outputs did not save training manifests. Same counts alone are not proof of identical training data.

Keep existing manuscript figures. The main deliverable is a compact comparison table, with case-level paired differences for interpretation. Compare preictal results only with the corrected nonoverlapping runs. Timing means condition on detected events and may describe different events for the two models.


In [1]:
from pathlib import Path
from collections import Counter
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path('/data/halders2/EEG_anomaly')
LINEAR_ROOT = PROJECT / 'chbmit_linear_baseline_results'
R4_ROOTS = {
    'ictal': PROJECT / 'chbmit_ictal_rossler_bd_v4_results',
    'preictal': PROJECT / 'chbmit_preictal_35to5_nonoverlap_rossler_bd_v4_results',
}
OUT = PROJECT / 'chbmit_linear_vs_R4_analysis'
TASKS = ['ictal', 'preictal']  # Use ['ictal'] while corrected preictal jobs finish.
EXPECTED_CASES = [f'chb{i:02d}' for i in range(1, 25)]
EXPECTED_SEEDS = [1107, 1249, 1388, 1523, 1697]
DATA_SEED = 17
MIN_R4_SEEDS = 1  # Match the primary convention: mean across successful seeds.
REQUIRE_ALL_CASES = True
PERSISTENCE = {'ictal': 3.0, 'preictal': 60.0}
SENSITIVITY = {'ictal': 'ictal_sensitivity', 'preictal': 'preictal_sensitivity'}
TIMING = {'ictal': 'ictal_delay_mean_seconds', 'preictal': 'warning_lead_mean_minutes'}
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 220)
print('Output:', OUT)
for task in TASKS:
    print(task, 'linear:', LINEAR_ROOT / task, 'R4:', R4_ROOTS[task])


Output: /data/halders2/EEG_anomaly/chbmit_linear_vs_R4_analysis
ictal linear: /data/halders2/EEG_anomaly/chbmit_linear_baseline_results/ictal R4: /data/halders2/EEG_anomaly/chbmit_ictal_rossler_bd_v4_results
preictal linear: /data/halders2/EEG_anomaly/chbmit_linear_baseline_results/preictal R4: /data/halders2/EEG_anomaly/chbmit_preictal_35to5_nonoverlap_rossler_bd_v4_results


## 1. Load the primary configurations

One linear fit per case is compared with that case's mean across available successful R4 seeds. Failed R4 runs are reported, not converted into zero sensitivity. Error or duplicate output files require inspection. All five R4 seeds are preferred; the retained seed count is exported for every case.


In [2]:
def load_summaries(root, r4_only=False):
    if not root.is_dir():
        raise FileNotFoundError(root)
    frames, issues = [], []
    for path in sorted(root.rglob('all_patient_summary.csv')):
        if r4_only and "_reg4_ds17_" not in str(path.parent):
            continue
        try:
            frame = pd.read_csv(path)
            frame['source_csv'] = str(path.resolve())
            frames.append(frame)
        except Exception as exc:
            issues.append({'file': str(path), 'issue': str(exc)})
    if issues:
        display(pd.DataFrame(issues))
        raise RuntimeError('Unreadable summaries: resolve before comparing.')
    if not frames:
        raise RuntimeError(f'No summaries found under {root}')
    return pd.concat(frames, ignore_index=True, sort=False)


def successful(frame):
    error = frame.get('error', pd.Series('', index=frame.index))
    return error.fillna('').astype(str).str.strip().eq('')


def filter_primary(frame, task, linear):
    mask = np.isclose(pd.to_numeric(frame['alarm_persistence_seconds'], errors='coerce'), PERSISTENCE[task])
    mask &= pd.to_numeric(frame['data_seed'], errors='coerce').eq(DATA_SEED)
    if linear:
        mask &= frame['model'].eq('direct_linear_ridge') & frame['task'].eq(task)
    else:
        mask &= pd.to_numeric(frame['regime_id'], errors='coerce').eq(4)
        reservoir_n = pd.to_numeric(frame['num_reservoirs'], errors='coerce')
        mask &= reservoir_n.eq(10) | (reservoir_n.isna() & ~successful(frame))
    return frame.loc[mask].copy()


linear_runs, r4_runs, inventory = {}, {}, []
for task in TASKS:
    lin = filter_primary(load_summaries(LINEAR_ROOT / task), task, True)
    res = filter_primary(load_summaries(R4_ROOTS[task], r4_only=True), task, False)
    if lin.duplicated(['patient']).any():
        raise ValueError(f'{task}: duplicate linear case results')
    if res.duplicated(['patient', 'res_seed']).any():
        raise ValueError(f'{task}: duplicate R4 case/seed results')
    if set(lin['patient']) - set(EXPECTED_CASES) or set(res['patient']) - set(EXPECTED_CASES):
        raise ValueError(f'{task}: unexpected case IDs')
    if set(pd.to_numeric(res['res_seed']).astype(int)) - set(EXPECTED_SEEDS):
        raise ValueError(f'{task}: unexpected R4 seeds')
    lin_ok, res_ok = lin.loc[successful(lin)].copy(), res.loc[successful(res)].copy()
    for case in EXPECTED_CASES:
        count = int(res_ok.loc[res_ok['patient'].eq(case), 'res_seed'].nunique())
        inventory.append({'task': task, 'case': case,
                          'linear_success': bool(lin_ok['patient'].eq(case).any()),
                          'r4_successful_seeds': count,
                          'r4_error_rows': int((res['patient'].eq(case) & ~successful(res)).sum())})
    linear_runs[task], r4_runs[task] = lin_ok, res_ok

inventory = pd.DataFrame(inventory)
inventory.to_csv(OUT / 'run_inventory.csv', index=False)
print(inventory.to_string(index=False))
if REQUIRE_ALL_CASES and ((~inventory['linear_success']).any() or (inventory['r4_successful_seeds'] < MIN_R4_SEEDS).any()):
    raise RuntimeError('Some requested cases are incomplete. Wait for runs, fix errors, or analyze a completed TASKS subset.')


    task  case  linear_success  r4_successful_seeds  r4_error_rows
   ictal chb01            True                    5              0
   ictal chb02            True                    5              0
   ictal chb03            True                    5              0
   ictal chb04            True                    5              0
   ictal chb05            True                    5              0
   ictal chb06            True                    5              0
   ictal chb07            True                    5              0
   ictal chb08            True                    5              0
   ictal chb09            True                    5              0
   ictal chb10            True                    5              0
   ictal chb11            True                    5              0
   ictal chb12            True                    5              0
   ictal chb13            True                    5              0
   ictal chb14            True                    5           

## 2. Check retained test identities and reported settings

The check compares the multiset of negative scan IDs and positive seizure IDs for the linear model against every included R4 seed, plus train/validation/test and positive sequence counts. The linear metadata supplies its requested preprocessing settings. A mismatch stops the comparison; do not select whichever model's cases look more favorable.

Reservoir test outputs identify windows but usually do not store the original training/validation manifest. Verify that the same source versions and EDF data generated both runs; this audit is not proof of historical training-sample identity. Preictal nonoverlap is also checked in the linear run's selected-window audit.


In [3]:
def artifact_file(row, filename):
    parent = Path(row['source_csv']).parent
    candidates = [parent / filename, parent / str(row['patient']) / filename]
    found = [p for p in candidates if p.is_file()]
    if len(found) != 1:
        raise FileNotFoundError(f'Expected one {filename} near {parent}; found {len(found)}')
    return found[0]


def ids_from_file(path, column):
    frame = pd.read_csv(path)
    if column not in frame or frame[column].isna().any():
        raise ValueError(f'Missing IDs in {path}')
    return Counter(frame[column].astype(str))

# Match chb12's linear test events to the original R4 evaluation.
# Original linear and reservoir output files are not modified.

import shutil

case = "chb12"
task = "ictal"

lin = linear_runs[task].loc[
    linear_runs[task]["patient"].eq(case)
].iloc[0].copy()

res = r4_runs[task].loc[
    r4_runs[task]["patient"].eq(case)
].sort_values("res_seed")

event_name = f"{case}_ictal_events_pers3p0s.csv"

# Require every R4 seed to use the same event IDs.
reference_ids = ids_from_file(
    artifact_file(res.iloc[0], event_name), "seizure_id"
)

for _, reservoir_row in res.iterrows():
    ids = ids_from_file(
        artifact_file(reservoir_row, event_name), "seizure_id"
    )
    if ids != reference_ids:
        raise ValueError("R4 seeds used different positive-event sets.")

events = pd.read_csv(artifact_file(lin, event_name))

if events["seizure_id"].duplicated().any():
    raise ValueError("Duplicate linear event IDs.")

matched = events.loc[
    events["seizure_id"].isin(reference_ids.keys())
].copy()

if Counter(matched["seizure_id"].astype(str)) != reference_ids:
    raise ValueError("Linear predictions are missing original R4 events.")

# Parse saved alarm flags explicitly.
alarm_text = matched["alarm"].astype(str).str.lower().str.strip()
if not alarm_text.isin(["true", "false", "1", "0"]).all():
    raise ValueError("Unrecognized alarm values.")

detected = alarm_text.isin(["true", "1"])
n_events = len(matched)
n_detected = int(detected.sum())

delays = pd.to_numeric(
    matched.loc[detected, "detection_delay_seconds"],
    errors="raise",
)

if not np.isfinite(delays).all():
    raise ValueError("A detected event has an invalid delay.")

row = lin.to_dict()
row.update({
    "positive_total": n_events,
    "positive_tp": n_detected,
    "positive_fn": n_events - n_detected,
    "ictal_total": n_events,
    "ictal_tp": n_detected,
    "ictal_fn": n_events - n_detected,
    "sensitivity": n_detected / n_events,
    "ictal_sensitivity": n_detected / n_events,
    "balanced_accuracy": (
        n_detected / n_events + float(lin["specificity"])
    ) / 2,
    "ictal_delay_mean_seconds": delays.mean(),
    "ictal_delay_median_seconds": delays.median(),
})

view_dir = OUT / "matched_linear_ictal_chb12"
view_dir.mkdir(parents=True, exist_ok=True)

source_dir = Path(lin["source_csv"]).parent

# Preserve the information needed by the existing pairing audit.
for filename in [
    "run_metadata.json",
    "selected_window_overlap_audit.csv",
    f"{case}_interictal_test_scans_pers3p0s.csv",
]:
    src = source_dir / filename
    dst = view_dir / filename
    if src.resolve() != dst.resolve():
        shutil.copy2(src, dst)

matched.to_csv(view_dir / event_name, index=False)

(view_dir / "comparison_note.txt").write_text(
    "Comparison-only subset: linear positive-event metrics recalculated "
    "on the original R4 seizure IDs. Predictor, calibration, and negative "
    "test results are unchanged. Original linear outputs are preserved.\n"
)

row["source_csv"] = str(
    (view_dir / "all_patient_summary.csv").resolve()
)
pd.DataFrame([row]).to_csv(row["source_csv"], index=False)

linear_runs[task] = pd.concat(
    [
        linear_runs[task].loc[
            ~linear_runs[task]["patient"].eq(case)
        ],
        pd.DataFrame([row]),
    ],
    ignore_index=True,
)

print("Matched chb12 events:", n_events)
print("Linear sensitivity on matched events:", row["ictal_sensitivity"])
print("Comparison-only files:", view_dir)

Matched chb12 events: 27
Linear sensitivity on matched events: 0.037037037037037035
Comparison-only files: /data/halders2/EEG_anomaly/chbmit_linear_vs_R4_analysis/matched_linear_ictal_chb12


In [4]:

checks = []
settings = ['segment_seconds', 'chunk_seconds', 'stride_seconds', 'pred_horizon',
            'washout', 'threshold_percentile']
for task in TASKS:
    linear = linear_runs[task].set_index('patient', drop=False)
    for _, row in r4_runs[task].iterrows():
        case = row['patient']
        if case not in linear.index:
            continue
        lin = linear.loc[case]
        suffix = str(float(PERSISTENCE[task])).replace('.', 'p')
        metadata_path = Path(lin['source_csv']).parent / 'run_metadata.json'
        metadata = json.loads(metadata_path.read_text())
        arguments = metadata['arguments']
        if task == 'preictal' and float(arguments['interictal_scan_stride_seconds']) != 1800:
            raise ValueError(f'{case}: preictal linear baseline is not using 1800-s selection stride')
        audit = pd.read_csv(Path(lin['source_csv']).parent / 'selected_window_overlap_audit.csv')
        overlap_ok = bool(audit['overlapping_pairs'].eq(0).all())
        fields = ['train_scans', 'val_scans', 'interictal_test_scans', f'{task}_total', 'pca_components']
        counts_ok = all(pd.notna(row.get(k)) and pd.notna(lin.get(k)) and float(row[k]) == float(lin[k]) for k in fields)
        settings_ok = all(pd.notna(row.get(k)) and float(row[k]) == float(arguments[k]) for k in settings)
        settings_ok &= bool(np.isclose(float(lin['ridge']), float(arguments['ridge'])))
        neg_name = f'{case}_interictal_test_scans_pers{suffix}s.csv'
        pos_name = f'{case}_{task}_events_pers{suffix}s.csv'
        neg_ok = ids_from_file(artifact_file(lin, neg_name), 'scan_id') == ids_from_file(artifact_file(row, neg_name), 'scan_id')
        pos_ok = ids_from_file(artifact_file(lin, pos_name), 'seizure_id') == ids_from_file(artifact_file(row, pos_name), 'seizure_id')
        checks.append({'task': task, 'case': case, 'r4_seed': int(row['res_seed']),
                       'counts_match': counts_ok, 'settings_match': settings_ok,
                       'negative_ids_match': neg_ok, 'positive_ids_match': pos_ok,
                       'linear_overlap_audit_passed': overlap_ok})

pairing_audit = pd.DataFrame(checks)
if pairing_audit.empty:
    raise RuntimeError('No matched cases available')
flag_cols = ['counts_match', 'settings_match', 'negative_ids_match', 'positive_ids_match', 'linear_overlap_audit_passed']
pairing_audit['passed'] = pairing_audit[flag_cols].all(axis=1)
pairing_audit.to_csv(OUT / 'pairing_audit.csv', index=False)
print(pairing_audit.groupby('task')['passed'].agg(['count', 'sum']).to_string())
if not pairing_audit['passed'].all():
    print(pairing_audit.loc[~pairing_audit['passed']].to_string(index=False))
    raise RuntimeError('Pairing mismatch. Inspect pairing_audit.csv before interpreting model differences.')
print('Available test IDs, counts, settings, and linear overlap audits agree.')
print('Historical R4 training-window identities remain unverified without saved training manifests.')


          count  sum
task                
ictal       120  120
preictal    120  120
Available test IDs, counts, settings, and linear overlap audits agree.
Historical R4 training-window identities remain unverified without saved training manifests.


## 3. Case-level paired comparisons and cohort summary

The primary table uses the same matched cases for both models. Each R4 value is a mean over successful reservoir realizations, and each linear value is one deterministic fit for the fixed split. Cases remain the descriptive units, including separate chb01 and chb21 cases; no independent-patient significance tests are performed here.

Timing is seconds for seizure-onset detection and minutes for preictal warning. Timing means are conditional on available detections for each model, so the contributing cases/events can differ. Paired timing differences below use only cases where both values are defined.


In [5]:
case_tables, cohort_rows, delta_rows = [], [], []
for task in TASKS:
    cols = [SENSITIVITY[task], 'specificity', 'balanced_accuracy', 'false_alarms_per_hour', TIMING[task]]
    metric_names = ['sensitivity', 'specificity', 'balanced_accuracy', 'FAR_per_hour', 'timing']
    rename = dict(zip(cols, metric_names))
    lin = linear_runs[task].set_index('patient')[cols].rename(columns=rename)
    r4 = r4_runs[task].groupby('patient')[cols].mean().rename(columns=rename)
    nseed = r4_runs[task].groupby('patient')['res_seed'].nunique()
    paired = lin.add_prefix('linear_').join(r4.add_prefix('R4_'), how='inner')
    paired = paired.join(nseed.rename('R4_successful_seeds'))
    if paired.empty:
        raise RuntimeError(f'{task}: no matched cases')
    for name in metric_names:
        for model in ['linear', 'R4']:
            x = pd.to_numeric(paired[f'{model}_{name}'], errors='raise')
            if np.isinf(x).any() or (name != 'timing' and x.isna().any()):
                raise ValueError(f'{task} {model} {name}: invalid values')
        paired[f'linear_minus_R4_{name}'] = paired[f'linear_{name}'] - paired[f'R4_{name}']
        delta = paired[f'linear_minus_R4_{name}'].dropna()
        delta_rows.append({'task': task, 'metric': name, 'n_paired_cases': len(delta),
                           'mean_linear_minus_R4': delta.mean(), 'median_linear_minus_R4': delta.median()})
    paired['task'] = task
    paired['timing_unit'] = 'seconds' if task == 'ictal' else 'minutes'
    case_tables.append(paired.reset_index())
    for model in ['R4', 'linear']:
        row = {'task': task, 'model': model, 'n_cases': len(paired),
               'n_individuals': len(set('chb01_chb21' if c in {'chb01', 'chb21'} else c for c in paired.index)),
               'timing_unit': paired['timing_unit'].iloc[0],
               'n_cases_with_timing': paired[f'{model}_timing'].notna().sum()}
        row.update({name: paired[f'{model}_{name}'].mean() for name in metric_names})
        cohort_rows.append(row)

case_comparison = pd.concat(case_tables, ignore_index=True)
cohort_comparison = pd.DataFrame(cohort_rows)
paired_differences = pd.DataFrame(delta_rows)
case_comparison.to_csv(OUT / 'linear_vs_R4_case_comparison.csv', index=False)
cohort_comparison.to_csv(OUT / 'linear_vs_R4_cohort_summary.csv', index=False)
paired_differences.to_csv(OUT / 'linear_vs_R4_paired_differences.csv', index=False)
print('COHORT COMPARISON: CASE-LEVEL MEANS')
print(cohort_comparison.to_string(index=False))
print('\nPAIRED CASE DIFFERENCES: LINEAR MINUS R4')
print(paired_differences.to_string(index=False))
print('\nCASE-LEVEL COMPARISON')
print(case_comparison.to_string(index=False))
print('\nSaved tables to:', OUT)


COHORT COMPARISON: CASE-LEVEL MEANS
    task  model  n_cases  n_individuals timing_unit  n_cases_with_timing  sensitivity  specificity  balanced_accuracy  FAR_per_hour    timing
   ictal     R4       24             23     seconds                   23     0.721695     0.872778           0.797236     16.333333 12.188274
   ictal linear       24             23     seconds                   22     0.534108     0.869444           0.701776     18.500000 13.667965
preictal     R4       24             23     minutes                   16     0.406647     0.754383           0.580515      0.643461 30.196354
preictal linear       24             23     minutes                   16     0.422619     0.715116           0.568868      0.713574 29.020833

PAIRED CASE DIFFERENCES: LINEAR MINUS R4
    task            metric  n_paired_cases  mean_linear_minus_R4  median_linear_minus_R4
   ictal       sensitivity              24             -0.187587               -0.128571
   ictal       specificity        

In [6]:
# Supplementary table from the completed cohort comparison.
table = cohort_comparison.copy()

table["Analysis"] = table["task"].map({
    "ictal": "Seizure-onset",
    "preictal": "Preictal",
})
table["Model"] = table["model"].map({
    "R4": "R4",
    "linear": "Linear ridge",
})

for source, label, digits in [
    ("sensitivity", "Sensitivity", 3),
    ("specificity", "Specificity", 3),
    ("balanced_accuracy", "Balanced accuracy", 3),
    ("FAR_per_hour", "FAR/h", 3),
    ("timing", "Mean timing", 2),
]:
    table[label] = table[source].map(
        lambda value, d=digits: f"{value:.{d}f}"
    )

table["Timing unit"] = table["timing_unit"]
table["Timing cases"] = table["n_cases_with_timing"].astype(int)

table = table[[
    "Analysis",
    "Model",
    "Sensitivity",
    "Specificity",
    "Balanced accuracy",
    "FAR/h",
    "Mean timing",
    "Timing unit",
    "Timing cases",
]]

display(table)

,Analysis,Model,Sensitivity,Specificity,Balanced accuracy,FAR/h,Mean timing,Timing unit,Timing cases
0,Seizure-onset,R4,0.722,0.873,0.797,16.333,12.19,seconds,23
1,Seizure-onset,Linear ridge,0.534,0.869,0.702,18.500,13.67,seconds,22
2,Preictal,R4,0.407,0.754,0.581,0.643,30.20,minutes,16
3,Preictal,Linear ridge,0.423,0.715,0.569,0.714,29.02,minutes,16


In [7]:
# Eligible preictal events and contributing cases/individuals.
# Reads existing outputs only; no models are rerun.

pre_runs = r4_runs["preictal"].copy()

if pre_runs.duplicated(["patient", "res_seed"]).any():
    raise ValueError("Duplicate preictal case/seed results.")

if set(pre_runs["patient"]) != set(EXPECTED_CASES):
    raise ValueError("Expected results for all 24 cases.")

rows = []

for case in EXPECTED_CASES:
    runs = pre_runs.loc[
        pre_runs["patient"].eq(case)
    ].sort_values("res_seed")

    seeds = set(pd.to_numeric(runs["res_seed"]).astype(int))
    if seeds != set(EXPECTED_SEEDS):
        raise ValueError(f"{case}: not all five R4 seeds are available.")

    reference_ids = None
    warned_ids = set()
    warning_counts = []
    seed_lead_means = []

    for _, run in runs.iterrows():
        path = artifact_file(
            run, f"{case}_preictal_events_pers60p0s.csv"
        )
        events = pd.read_csv(path)

        required = {
            "seizure_id", "alarm", "warning_lead_time_minutes"
        }
        if not required.issubset(events.columns):
            raise ValueError(f"Missing required columns in {path}")

        if (
            events.empty
            or events["seizure_id"].isna().any()
            or events["seizure_id"].duplicated().any()
        ):
            raise ValueError(f"Missing or duplicate event IDs in {path}")

        ids = set(events["seizure_id"].astype(str))
        if reference_ids is None:
            reference_ids = ids
        elif ids != reference_ids:
            raise ValueError(f"{case}: event sets differ across seeds.")

        if len(events) != int(run["preictal_total"]):
            raise ValueError(f"{case}: event count disagrees with summary.")

        alarm_text = events["alarm"].astype(str).str.strip().str.lower()
        if not alarm_text.isin(["true", "false", "1", "0"]).all():
            raise ValueError(f"Unrecognized alarm values in {path}")

        detected = alarm_text.isin(["true", "1"])
        leads = pd.to_numeric(
            events["warning_lead_time_minutes"], errors="raise"
        )

        if not np.isfinite(leads.loc[detected]).all():
            raise ValueError(f"{case}: detected event has invalid lead time.")
        if (leads.loc[detected] <= 0).any():
            raise ValueError(f"{case}: nonpositive warning lead time.")
        if leads.loc[~detected].notna().any():
            raise ValueError(f"{case}: missed event has a lead time.")

        n_detected = int(detected.sum())
        if n_detected != int(run["preictal_tp"]):
            raise ValueError(f"{case}: warning count disagrees with summary.")

        if not np.isclose(
            n_detected / len(events),
            float(run["preictal_sensitivity"]),
        ):
            raise ValueError(f"{case}: sensitivity disagrees with events.")

        seed_mean = leads.loc[detected].mean()
        if not np.isclose(
            seed_mean,
            float(run["warning_lead_mean_minutes"]),
            equal_nan=True,
        ):
            raise ValueError(f"{case}: lead time disagrees with summary.")

        warned_ids.update(
            events.loc[detected, "seizure_id"].astype(str)
        )
        warning_counts.append(n_detected)
        seed_lead_means.append(seed_mean)

    rows.append({
        "Case": case,
        "Individual": (
            "chb01/chb21" if case in {"chb01", "chb21"} else case
        ),
        "Eligible events": len(reference_ids),
        "Successful R4 realizations": len(runs),
        "Realizations with warnings": sum(n > 0 for n in warning_counts),
        "Events warned in at least one realization": len(warned_ids),
        "Contributes lead time": any(n > 0 for n in warning_counts),
    })

preictal_event_table = pd.DataFrame(rows)

with_timing = preictal_event_table.loc[
    preictal_event_table["Contributes lead time"]
]

contribution_counts = pd.DataFrame([
    {
        "Analysis": "Preictal sensitivity",
        "Cases": len(preictal_event_table),
        "Individuals": preictal_event_table["Individual"].nunique(),
    },
    {
        "Analysis": "Conditional warning lead time",
        "Cases": len(with_timing),
        "Individuals": with_timing["Individual"].nunique(),
    },
])

preictal_event_table.to_csv(
    OUT / "preictal_R4_eligible_events_by_case.csv", index=False
)
contribution_counts.to_csv(
    OUT / "preictal_R4_contributing_cases_individuals.csv", index=False
)

print("PREICTAL R4 ELIGIBLE-EVENT TABLE")
display(preictal_event_table)

print("\nCONTRIBUTING CASES AND INDIVIDUALS")
display(contribution_counts)

print(
    "\nTotal eligible events, counted once per case:",
    int(preictal_event_table["Eligible events"].sum()),
)
print(
    "Total successful R4 realizations:",
    int(preictal_event_table["Successful R4 realizations"].sum()),
)

print(
    "\nAggregation: sensitivity is calculated across eligible events within "
    "each realization, averaged across realizations within each case, "
    "then averaged across cases."
)
print(
    "Lead times are averaged across warned events within each realization, "
    "then across realizations with defined timing within each case, "
    "then across cases with defined timing."
)
print(
    "'Events warned in at least one realization' counts unique events "
    "across the five realizations; it is descriptive and is not the "
    "denominator or numerator used for mean sensitivity."
)
print("\nSaved tables to:", OUT)

PREICTAL R4 ELIGIBLE-EVENT TABLE


,Case,Individual,Eligible events,Successful R4 realizations,Realizations with warnings,Events warned in at least one realization,Contributes lead time
0,chb01,chb01/chb21,1,5,0,0,False
1,chb02,chb02,2,5,5,1,True
2,chb03,chb03,2,5,0,0,False
3,chb04,chb04,2,5,4,1,True
4,chb05,chb05,3,5,2,1,True
5,chb06,chb06,7,5,5,1,True
6,chb07,chb07,3,5,5,3,True
7,chb08,chb08,4,5,5,4,True
8,chb09,chb09,3,5,5,1,True
9,chb10,chb10,6,5,5,2,True



CONTRIBUTING CASES AND INDIVIDUALS


,Analysis,Cases,Individuals
0,Preictal sensitivity,24,23
1,Conditional warning lead time,16,16



Total eligible events, counted once per case: 64
Total successful R4 realizations: 120

Aggregation: sensitivity is calculated across eligible events within each realization, averaged across realizations within each case, then averaged across cases.
Lead times are averaged across warned events within each realization, then across realizations with defined timing within each case, then across cases with defined timing.
'Events warned in at least one realization' counts unique events across the five realizations; it is descriptive and is not the denominator or numerator used for mean sensitivity.

Saved tables to: /data/halders2/EEG_anomaly/chbmit_linear_vs_R4_analysis
